In [ ]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
%%writefile missing.sh

# Calculate variant and sample missingness
plink2 \
    --bfile "${GENO}/EUR_passed_chr${CHROM}" \
    --missing \
    --memory 3100 \
    --out "${OUTPUT}/EUR_passed_chr${CHROM}"


In [ ]:
# Define path to logging directory
LOGGING=f"{os.environ['WORKSPACE_BUCKET']}/data/logging"
# Define path to output directory 
OUTPUT=f"{os.environ['WORKSPACE_BUCKET']}/data/chip/plink"
# Define mount path to genomic data 
MNT_PATH="/mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/chip/plink"
# Define virtual machine disk size 
DISK_SIZE = 4 
    
# Loop through chromosomes 
for CHROM in [2, 4, 15, 17]:    
    
    print(f"Submitting dsub command for chromosome {CHROM}")

    # Define path to genomic data 
    BED=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_passed_chr{CHROM}.bed'
    BIM=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_passed_chr{CHROM}.bim'
    FAM=f'gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_passed_chr{CHROM}.fam'
    
    # Sumbission command for dsub 
    command = [
        "aou_dsub",
        "--project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--user-project", os.getenv('GOOGLE_CLOUD_PROJECT'), 
        "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/gfanz/plink2:latest",
        "--disk-size", str(DISK_SIZE),
        "--machine-type", "n1-standard-1",
        "--logging", LOGGING,
        "--input", f"BED={BED}",
        "--input", f"BIM={BIM}",
        "--input", f"FAM={FAM}",
        "--output-recursive", f"OUTPUT={OUTPUT}",
        "--env", f"GENO={MNT_PATH}",
        "--env", f"CHROM={CHROM}",
        "--script", "missing.sh"
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {' '.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

In [ ]:
# Load packages 
import os
import subprocess
import pandas as pd 
from functools import reduce
from io import StringIO
from pathlib import Path
import shutil
import gc 

# Loop through chromosomes 
for CHROM in [2, 4, 15, 17]:

    # Load missingness statistics before QC
    PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/chip/plink/EUR_passed_chr{CHROM}.vmiss"
    result = subprocess.run(
        ["gcloud", "storage", "cat", PATH,
         f"--billing-project={os.environ['GOOGLE_CLOUD_PROJECT']}"],
        capture_output=True, text=True, check=True
    )
    df1 = pd.read_csv(StringIO(result.stdout), sep="\t")[['ID', 'MISSING_CT']]

    # Load missingness statistics after QC
    PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/chip/plink/EUR_ab0.35_dp20_call0.9_hwe1e15_chr{CHROM}.vmiss"
    result = subprocess.run(
        ["gcloud", "storage", "cat", PATH,
         f"--billing-project={os.environ['GOOGLE_CLOUD_PROJECT']}"],
        capture_output=True, text=True, check=True
    )
    df2 = pd.read_csv(StringIO(result.stdout), sep="\t")[['ID', 'MISSING_CT']]
    
    # Rename columns and harmonize variant identifiers
    df2.rename(columns={'MISSING_CT':'MISSING_CT_FILTER'}, inplace=True)
    df2['ID'] = 'chr' + df2['ID']
    
    # Compare missingness counts for matching variants
    df = pd.merge(df1, df2, on=['ID'])
    
    # Check that missingness has not decreased after QC
    check = len(df[df['MISSING_CT'] > df['MISSING_CT_FILTER']]) 
    
    if check > 0: 
        print("ERROR") 
    
    # Confirm that missingness increased for at least one variant
    check = len(df[df['MISSING_CT'] < df['MISSING_CT_FILTER']]) 
    
    if not check > 0: 
        print("ERROR") 
        
    print(F"Chromosome {CHROM} was filtered correctly ....") 
        
print("All chromosomes were filtered correctly ....") 
    